# APS-2 Regressão

## - Guilherme Guedes
## - Luigi Sibinelli

In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor # pode dar erro de 32 -> 64 bits!
                                 # instalar "brew install libomp"
                                 # (só pra MAC isso ludão)

from sklearn.model_selection import cross_val_score, RepeatedKFold, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## 1. Carregamento e Pré-processamento

### 1.1 Carregamento dos dados

Carregar o dataset Adult Census Income via `ucimlrepo`.

In [51]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
adult = fetch_ucirepo(id=2) 

In [52]:
X = adult.data.features
y = adult.data.targets

df = pd.concat([X, y], axis=1)

df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


### 1.2 Inspeção inicial

Verificar o `shape`, `tipos de dados`, `primeiras linhas` e `valores ausentes`.

" copiar " do Projeto 1 esse começo msm

In [53]:
# shape
df.shape

(48842, 15)

In [54]:
# tratando a coluna do target para remover inconsistência
df["income"] = df["income"].replace({"<=50K.": "<=50K", ">50K.": ">50K"})

#EXPLICITANDO A QT DE INSTÂNCIAS
income_counts = df["income"].value_counts()
print(income_counts)
print(f"\nSoma total: {income_counts.sum()}")

income
<=50K    37155
>50K     11687
Name: count, dtype: int64

Soma total: 48842


In [55]:
# tipos de dados
df.dtypes.astype(str).sort_values()

age                int64
fnlwgt             int64
education-num      int64
capital-gain       int64
capital-loss       int64
hours-per-week     int64
workclass         object
education         object
marital-status    object
occupation        object
relationship      object
race              object
sex               object
native-country    object
income            object
dtype: object

In [56]:
# primeiras linhas
df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [57]:
# valores ausentes

### Verificando e informando quantidade de valores faltantes nas features (considerando '?' e NaN)
## OBS: valor absoluto e porcentual 

features_df = df.drop(columns=["income"]).copy()

is_question_mark = features_df.apply(
    lambda col: col.astype(str).str.strip().eq("?") if col.dtype == "object" else pd.Series(False, index=col.index)
)
is_nan = features_df.isna()

missing_mask = is_question_mark | is_nan
missing_counts = missing_mask.sum().sort_values(ascending=False)
missing_counts = missing_counts[missing_counts > 0]

if missing_counts.empty:
    print("Nenhuma feature com valores faltantes ('?' ou NaN).")
else:
    missing_summary = pd.DataFrame({
        "missing_count": missing_counts,
        "missing_pct": (missing_counts / len(features_df) * 100).round(2)
    })
    display(missing_summary)

,missing_count,missing_pct
occupation,2809,5.75
workclass,2799,5.73
native-country,857,1.75


### 1.3 Definição de X e y
Como o objetivo é prever `hours-per-week`, separamos essa coluna como target `y` e removemos ela de `X`.

In [58]:
# blablabla

# columns_to_remove = [
#     "hours-per-week",
#     "capital-loss",
#     "capital-gain",
#     "fnlwgt",
#     "education",
# ]

# df = df.drop(columns=columns_to_remove, errors="ignore")
# print("Features removidas:", columns_to_remove)




In [59]:
X = df.drop('hours-per-week', axis=1).copy()
y = df['hours-per-week'].copy() #target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

In [60]:
# Separando colunas numéricas e categóricas para análise posterior
num_cols = X_train.select_dtypes(include=["number"]).columns.tolist()

cat_cols = X_train.select_dtypes(exclude=["number"]).columns.tolist()

### 1.4 Pipeline de pré-processamento

Definir um `ColumnTransformer` com dois sub-pipelines:
- **Numérico**: imputação pela mediana + `StandardScaler`
- **Categórico**: imputação pela moda + `OneHotEncoder`

In [61]:
# usar mesmo pipeline de pré processamento definido na aps1
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

In [62]:
preprocessor = ColumnTransformer(
    transformers=[
        ('numerical', numeric_transformer, num_cols),
        ('categorical', categorical_transformer, cat_cols)
    ]
)

In [ ]:
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   KNeighborsClassifier(n_neighbors=8)) #precisa mudar isso, eu suponho
])

NameError: name 'KNeighborsClassifier' is not defined

## 2. Feature Engineering

Criar ao menos 4 novas features a partir das variáveis existentes para enriquecer o modelo:

- `capital_net` = `capital-gain` − `capital-loss`
- `education_level`: agrupamento ordinal de `education-num`
- `work_experience_proxy` = `age` − `education-num` − 6
- Interações relevantes (ex.: `age` × `hours-per-week`)

In [ ]:
# pensar junto

## 3. Modelagem

Treinar 3 modelos distintos usando `Pipeline` com o pré-processamento definido anteriormente e validação cruzada `RepeatedKFold` (5×):

- **Baseline**: Ridge / Lasso / Linear Regression
- **Ensemble 1**: Random Forest Regressor
- **Ensemble 2**: XGBoost Regressor

In [ ]:
#incluir no pipe as novas features e treinar!!

## 4. Otimização e Seleção de Modelo

Realizamos hyperparameter tuning com `GridSearchCV` nos modelos mais promissores, comparando os resultados para escolher o melhor modelo.

In [ ]:
# blablabal

## 5. Análise de Métricas

Avaliamos o desempenho dos modelos com as métricas: RMSE, MAE e R². Comparamos os resultados entre todos os modelos treinados.

In [ ]:
# blablabl

## 6. Interpretação e Explicabilidade

Analisamos quais features mais impactam as previsões do melhor modelo, usando importância de features (feature importances do Random Forest / XGBoost) e análise de resíduos.

In [ ]:
# blablabla

## 7. Discussão Crítica e Conclusões

Discutir as limitações do modelo treinado, como multimodalidade da variável alvo, heterocedasticidade e questões de causalidade vs. correlação.